# 03 Dashboard export

## Cosa fa / Cosa NON fa

- prepara un export di esempio partendo da un mart disponibile
- non scrive file finché `EXPORT = False`
- se non trova colonne adatte, esporta un campione generico come fallback

In [ ]:
from pathlib import Path
import duckdb

ROOT = Path('.').resolve()
EXPORT = False
OUT_DIR = (ROOT / '..' / '_tmp').resolve()
TABLE_NAME = 'project_summary'

def find_mart(table_name):
    for base in [(ROOT / '..' / 'data' / 'mart').resolve(), (ROOT / '..' / '_runs').resolve()]:
        if not base.exists():
            continue
        matches = sorted(base.glob(f'**/*{table_name}*.parquet'))
        if matches:
            return matches[0]
    return None

mart_path = find_mart(TABLE_NAME)
OUT_DIR

In [ ]:
con = duckdb.connect()

def read_schema(path):
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    schema.columns = [str(col).lower() for col in schema.columns]
    return schema

def choose_columns(schema):
    name_col = 'column_name' if 'column_name' in schema.columns else schema.columns[0]
    type_col = 'column_type' if 'column_type' in schema.columns else schema.columns[1]
    rows = [
        {'name': str(row[name_col]), 'type': str(row[type_col]).upper()}
        for _, row in schema.iterrows()
    ]
    year_col = next((r['name'] for r in rows if r['name'].lower() == 'year' or 'anno' in r['name'].lower()), None)
    numeric_rows = [r for r in rows if any(token in r['type'] for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT'])]
    metric_col = next((r['name'] for r in numeric_rows if any(token in r['name'].lower() for token in ['value', 'tot', 'importo', 'ammontare', 'pct', 'percent'])), None)
    if metric_col is None and numeric_rows:
        metric_col = numeric_rows[0]['name']
    return year_col, metric_col

if mart_path:
    schema_df = read_schema(str(mart_path))
    YEAR_COL, METRIC_COL = choose_columns(schema_df)
    print({'YEAR_COL': YEAR_COL, 'METRIC_COL': METRIC_COL})
else:
    print('No mart parquet found. Run the pipeline first or update TABLE_NAME.')

In [ ]:
if mart_path and YEAR_COL and METRIC_COL:
    export_df = con.execute(
        f"SELECT {YEAR_COL} AS year_like, {METRIC_COL} AS metric_value FROM read_parquet('{mart_path}') ORDER BY 1"
    ).df()
elif mart_path:
    print('Using generic fallback export: no year-like or metric column detected.')
    export_df = con.execute(f"SELECT * FROM read_parquet('{mart_path}') LIMIT 1000").df()
else:
    export_df = None

if export_df is not None:
    display(export_df.head())

In [ ]:
if EXPORT and export_df is not None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = OUT_DIR / f'{TABLE_NAME}_dashboard.csv'
    parquet_path = OUT_DIR / f'{TABLE_NAME}_dashboard.parquet'
    export_df.to_csv(csv_path, index=False)
    con.register('export_df_view', export_df)
    con.execute(f"COPY (SELECT * FROM export_df_view) TO '{parquet_path}' (FORMAT PARQUET)")
    print(csv_path)
    print(parquet_path)
else:
    print('Export disabled. Set EXPORT = True to write files into ../_tmp/.')